
###  Robust Two-Regime Ensemble

---

**Author:** Santosh Bogati  
```

---
## SECTION 1 — Import Libraries

In [ ]:
# ── Core data manipulation ──────────────────────────────────────────────────
import pandas as pd          # DataFrames: load, inspect, filter data
import numpy as np           # Numerical operations (arrays, percentiles, etc.)

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
plt.rcParams['figure.facecolor'] = '#F8F9FA'
plt.rcParams['axes.facecolor']   = '#FFFFFF'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

# ── Scikit-learn: preprocessing ─────────────────────────────────────────────
from sklearn.impute     import SimpleImputer   # Fill missing values
from sklearn.preprocessing import StandardScaler  # Normalise feature scale

# ── Scikit-learn: model ─────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression  # Our winning model

# ── Scikit-learn: evaluation ────────────────────────────────────────────────
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics         import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

---
## SECTION 2 — Load Data




In [ ]:
# ── CHANGE PATHS if your files are in a different location ──────────────────
train = pd.read_csv('/content/Training file.csv')
test  = pd.read_csv('/content/Test file.csv')

# Define our feature list — x0 through x14 (15 features total)
features = [f'x{i}' for i in range(15)]

# Extract the target values as a numpy array
y = train['target'].values

print(f'Training set : {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'Test set     : {test.shape[0]:,} rows × {test.shape[1]} columns')
print(f'Features     : {features}')
print()
print('First 5 rows of training data:')
train.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## SECTION 3 — Exploratory Data Analysis (EDA)




### 3A — Basic Statistics

In [ ]:
print('=== TARGET STATISTICS ===')
print(f'  Min    : {y.min():.2f}')
print(f'  Mean   : {y.mean():.2f}')
print(f'  Median : {np.median(y):.2f}')
print(f'  Max    : {y.max():.2f}')
print(f'  Std    : {y.std():.2f}')
print()
print('Note: Mean (81) >> Median (7) — strong right skew due to extreme values')
print()
train[features].describe().round(2)

### 3B — Missing Values

In [ ]:
# Count missing values per column
missing_train = train[features].isnull().sum()
missing_pct   = (missing_train / len(train) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_train,
    'Missing %'    : missing_pct
})

print('Missing values in training set:')
print(missing_df.to_string())
print()
print(f'Total missing cells: {missing_train.sum()} out of {len(train)*15} ({missing_train.sum()/(len(train)*15)*100:.2f}%)')
print('All features have ~1% missing — very manageable with median imputation.')

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(missing_pct.index, missing_pct.values, color='#1C7293', width=0.6, edgecolor='none')
ax.set_title('Missing Values per Feature (%)', fontsize=14, fontweight='bold', pad=10)
ax.set_ylabel('Missing %', fontsize=11)
ax.set_ylim(0, 4)
for bar, val in zip(bars, missing_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'{val:.1f}%', ha='center', fontsize=9, color='#2C3E50')
ax.axhline(2.0, color='#E74C3C', linestyle='--', linewidth=1, alpha=0.5, label='2% reference')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print('All features have <2% missing. Strategy: fill with column MEDIAN (robust to outliers).')

### 3C — Target Distribution & Outlier Analysis

This is the **most important discovery** in the entire analysis.  
We use the **IQR (Interquartile Range)** method to identify outliers:
.

In [ ]:
# IQR outlier detection
q1  = np.percentile(y, 25)
q3  = np.percentile(y, 75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask_normal  = (y >= lower) & (y <= upper)
mask_extreme = ~mask_normal

print('=== IQR OUTLIER ANALYSIS ===')
print(f'  Q1 (25th percentile) : {q1:.2f}')
print(f'  Q3 (75th percentile) : {q3:.2f}')
print(f'  IQR                  : {iqr:.2f}')
print(f'  Normal range         : {lower:.2f}  to  {upper:.2f}')
print()
print(f'  Normal rows  : {mask_normal.sum()} ({mask_normal.mean()*100:.1f}%)')
print(f'  Extreme rows : {mask_extreme.sum()} ({mask_extreme.mean()*100:.1f}%)')
print()

# Variance breakdown
total_var   = np.sum((y - y.mean())**2)
extreme_var = np.sum((y[mask_extreme] - y.mean())**2)
print(f'  Variance from extreme rows: {extreme_var/total_var*100:.1f}% of TOTAL variance')
print()
print('  KEY FINDING: Only 7% of rows, but they contain 94.6% of all variance!')
print('  Any model that cannot predict these rows will score near R² = 0.')

In [ ]:
# Visualise the two regimes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full distribution
axes[0].hist(y[mask_normal],  bins=80, color='#1C7293', edgecolor='none', alpha=0.85, label=f'Normal rows (n={mask_normal.sum()})')
axes[0].hist(y[mask_extreme], bins=30, color='#E74C3C', edgecolor='none', alpha=0.85, label=f'Extreme rows (n={mask_extreme.sum()})')
axes[0].set_title('Target Distribution — Two Regimes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target Value', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend(fontsize=10)

# Right: x1 vs target
imp_tmp = SimpleImputer(strategy='median')
X_tmp   = pd.DataFrame(imp_tmp.fit_transform(train[features]), columns=features)
x1_range = np.linspace(-40, 40, 300)

axes[1].scatter(X_tmp['x1'][mask_normal],  y[mask_normal],  s=4,  alpha=0.25, color='#1C7293', label='Normal rows')
axes[1].scatter(X_tmp['x1'][mask_extreme], y[mask_extreme], s=20, alpha=0.8,  color='#E74C3C', label='Extreme rows')
axes[1].set_title('Target vs x1 — Clear Two-Pattern Structure', fontsize=14, fontweight='bold')
axes[1].set_xlabel('x1', fontsize=11)
axes[1].set_ylabel('target', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Confirm: extreme rows follow target ≈ 5 × x1²
fig, ax = plt.subplots(figsize=(8, 5))

x1_sq_ext = X_tmp['x1'][mask_extreme]**2
ax.scatter(x1_sq_ext, y[mask_extreme], s=25, alpha=0.7, color='#E74C3C', label='Extreme rows (actual)')

xline = np.linspace(0, 1100, 300)
ax.plot(xline, 5*xline, color='#F39C12', linewidth=2.5, label='y = 5 × x1²  (R² = 0.96)')

ax.set_title('Extreme Rows: target ≈ 5 × x1²', fontsize=14, fontweight='bold')
ax.set_xlabel('x1²', fontsize=11)
ax.set_ylabel('target', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Verify the R²
pred_formula = 5.0 * x1_sq_ext
print(f'R² of  "5 × x1²"  on extreme rows: {r2_score(y[mask_extreme], pred_formula):.4f}')
print()
print('The 5×x1² formula explains 96% of variance in extreme rows.')
print('The remaining 4% is captured by the linear base model.')

### 3D — Feature Correlations with Target

Since extreme rows dominate the raw data, we inspect correlations on **clean (normal) rows only**.  


In [ ]:
# Correlations on normal rows — the real signal
corrs_normal = pd.Series({
    f: X_tmp.loc[mask_normal, f].corr(pd.Series(y[mask_normal]))
    for f in features
}).sort_values()

print('Correlations with target (normal rows only):')
print(corrs_normal.round(4).to_string())
print()
print('x4 has r = 0.744 — extremely strong linear relationship!')
print('x5 has r = -0.371, x8 has r = -0.277, x2 has r = -0.248')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E74C3C' if v < 0 else '#27AE60' for v in corrs_normal]
bars = ax.barh(corrs_normal.index, corrs_normal.values, color=colors, edgecolor='none', height=0.6)
ax.axvline(0, color='#2C3E50', linewidth=1.2)
ax.set_title('Feature Correlations with Target (Normal Rows Only)', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Pearson Correlation', fontsize=11)
for bar, val in zip(bars, corrs_normal.values):
    offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()

---
## SECTION 4 — Preprocessing

### Median Imputation for extreme median handling


In [ ]:
# ── Step 1: Median Imputation ────────────────────────────────────────────────


imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(train[features]), columns=features)
X_test = pd.DataFrame(imp.transform(test[features]),      columns=features)

print('Missing values after imputation:')
print(f'  Train: {X.isnull().sum().sum()} (should be 0)')
print(f'  Test : {X_test.isnull().sum().sum()} (should be 0)')
print()
print('Medians used for imputation:')
for f, med in zip(features, imp.statistics_):
    print(f'  {f}: {med:.4f}')

---
## SECTION 5 — Feature Engineering (Most Important part here)

We added the **squared version of every feature** (x0², x1², ... x14²) bringing the total from 15 to 30 features.

**Why squared features?**
- The relationship between x1, specifically, and target is non-linear (parabolic for extreme rows).
- Adding x² allows a linear model to fit curved patterns.
- The linear model can now learn: `target ≈ a·x4 + b·x2 + c·x1²` — capturing both regimes.

The **same function** is applied to both train and test to ensure consistency.

In [ ]:
def build_features(df):
    """
    Creates engineered features:
      - All 15 original features (raw)
      - All 15 features squared (non-linear signal)
    Total: 30 features
    """
    out = df.copy()
    for f in features:
        out[f'{f}_sq'] = df[f] ** 2   # x0_sq, x1_sq, ..., x14_sq
    return out

X_eng      = build_features(X)
X_test_eng = build_features(X_test)

print(f'Features before engineering : {X.shape[1]}')
print(f'Features after  engineering : {X_eng.shape[1]}')
print()
print('New features added:')
new_feats = [c for c in X_eng.columns if c not in features]
print('  ', new_feats)

---
## SECTION 6 — Model Training

### Key Decision: Train on Normal Rows Only



In [ ]:
# ── Step 1: Identify normal rows (IQR method) ───────────────────────────────
q1, q3 = np.percentile(y, 25), np.percentile(y, 75)
iqr     = q3 - q1

# Normal: within 1.5×IQR of Q1 and Q3
mask_normal = (y >= q1 - 1.5*iqr) & (y <= q3 + 1.5*iqr)

print(f'Normal rows used for training: {mask_normal.sum()}')
print(f'Normal row target range      : {y[mask_normal].min():.2f} to {y[mask_normal].max():.2f}')
print(f'Extreme rows excluded        : {(~mask_normal).sum()} (handled by x1² formula)')

In [ ]:
# ── Step 2: Train LinearRegression on normal rows only ──────────────────────

# StandardScaler normalises all 30 features to mean=0, std=1
# This is critical — without it, large-range features would dominate
sc_final = StandardScaler()
m_final  = LinearRegression()

# fit_transform: learns mean & std from normal training rows, then scales
X_normal_scaled = sc_final.fit_transform(X_eng[mask_normal])

# Fit the linear model: minimises sum of squared errors on normal rows
m_final.fit(X_normal_scaled, y[mask_normal])

# Training R² on normal rows — should be ~0.91
train_pred_normal = m_final.predict(X_normal_scaled)
train_r2_normal   = r2_score(y[mask_normal], train_pred_normal)

print(f'LinearRegression R² on normal rows: {train_r2_normal:.4f}')
print()

# Show top coefficients
coef_df = pd.DataFrame({
    'Feature'    : X_eng.columns,
    'Coefficient': m_final.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Top 10 most important features by |coefficient|:')
print(coef_df.head(10).to_string(index=False))

---
## SECTION 7 — Cross Validation



In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Run 5-fold CV for the final ensemble
oof_preds  = np.zeros(len(y))   # out-of-fold predictions
fold_r2s   = []

for fold, (tr, val) in enumerate(kf.split(X_eng)):
    # Identify normal rows within THIS training fold
    mask_tr_norm = mask_normal[tr]

    # Train scaler and model on normal rows of this fold's training set
    sc = StandardScaler()
    m  = LinearRegression()
    m.fit(sc.fit_transform(X_eng.iloc[tr][mask_tr_norm]), y[tr][mask_tr_norm])

    # Predict on validation fold (never seen during training)
    p_clean = m.predict(sc.transform(X_eng.iloc[val]))
    p_x1sq  = 5.0 * X.iloc[val]['x1'].values**2

    # Blend: 90% linear model + 10% x1² formula
    blend = 0.9 * p_clean + 0.1 * p_x1sq
    oof_preds[val] = blend

    fr2 = r2_score(y[val], blend)
    fold_r2s.append(fr2)
    print(f'  Fold {fold+1}: R² = {fr2:.4f}')

print()
print(f'Mean CV R²  : {np.mean(fold_r2s):.4f}  ← HONEST competition estimate')
print(f'Std  CV R²  : {np.std(fold_r2s):.4f}')
print(f'CV   RMSE   : {np.sqrt(mean_squared_error(y, oof_preds)):.2f}')
print(f'CV   MAE    : {mean_absolute_error(y, oof_preds):.2f}')

In [ ]:
# Actual vs Predicted visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Actual vs Predicted scatter
axes[0].scatter(y[mask_normal],  oof_preds[mask_normal],  s=4,  alpha=0.3, color='#1C7293', label='Normal rows')
axes[0].scatter(y[~mask_normal], oof_preds[~mask_normal], s=20, alpha=0.7, color='#E74C3C', label='Extreme rows')
lims = [min(y.min(), oof_preds.min()), max(y.max(), oof_preds.max())]
axes[0].plot(lims, lims, 'k--', lw=1.5, label='Perfect fit')
axes[0].set_title('Actual vs Predicted (Out-of-Fold CV)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual target', fontsize=11)
axes[0].set_ylabel('Predicted target', fontsize=11)
axes[0].legend(fontsize=10)

# Right: Residuals distribution
resid = y - oof_preds
axes[1].hist(resid[mask_normal],  bins=60, color='#1C7293', alpha=0.8, edgecolor='none', label='Normal rows')
axes[1].hist(resid[~mask_normal], bins=20, color='#E74C3C', alpha=0.8, edgecolor='none', label='Extreme rows')
axes[1].axvline(0, color='black', lw=1.5)
axes[1].set_title('Residuals Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual − Predicted)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## SECTION 8 — Train Final Model & Generate Submission

.  


In [ ]:
# ═══════════════════════════════════════════════════════════════
#
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# ── 1. Load data ─────────────────────────────────────────────────────────────
train = pd.read_csv('/content/Training file.csv')
test  = pd.read_csv('/content/Test file.csv')
features = [f'x{i}' for i in range(15)]
y = train['target'].values

# ── 2. Impute missing values with MEDIAN ─────────────────────────────────────
# Median is robust to outliers — safer than mean when extreme values exist
imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(train[features]), columns=features)
X_test = pd.DataFrame(imp.transform(test[features]),      columns=features)

# ── 3. Feature engineering: add squared features ─────────────────────────────
# Squaring features allows the linear model to capture non-linear patterns
# e.g., target ≈ 5×x1² for extreme rows — only capturable if x1_sq is a feature
def build_features(df):
    out = df.copy()
    for f in features:
        out[f'{f}_sq'] = df[f] ** 2   # add x0_sq, x1_sq, ..., x14_sq
    return out

X_eng      = build_features(X)
X_test_eng = build_features(X_test)

# ── 4. Identify NORMAL rows using IQR method ─────────────────────────────────
# Normal rows: target within Q1 − 1.5×IQR  to  Q3 + 1.5×IQR
# Extreme rows (7%) follow 5×x1² and would confuse the linear model
q1, q3 = np.percentile(y, 25), np.percentile(y, 75)
iqr     = q3 - q1
mask_normal = (y >= q1 - 1.5*iqr) & (y <= q3 + 1.5*iqr)

# ── 5. Train LinearRegression ONLY on normal rows ────────────────────────────
# Training on clean data gives R²=0.91 on those rows
# Training on ALL data gives R²≈0.007 — extreme rows destroy the fit
sc_final = StandardScaler()
m_final  = LinearRegression()
m_final.fit(
    sc_final.fit_transform(X_eng[mask_normal]),  # scale then fit
    y[mask_normal]                               # normal targets only
)

# ── 6. Generate blended predictions for TEST set ─────────────────────────────
# Component A: clean linear model prediction (strong on normal rows)
p_clean = m_final.predict(sc_final.transform(X_test_eng))

# Component B: 5×x1² formula (captures extreme row pattern)
p_x1sq  = 5.0 * X_test['x1'].values**2

# Blend: 90% linear + 10% x1² formula
# (0.90 weight found by searching all blends 0.0→1.0 via cross-validation)
final_preds = 0.9 * p_clean + 0.1 * p_x1sq

print(f'Test prediction range : {final_preds.min():.2f}  to  {final_preds.max():.2f}')
print(f'Test prediction mean  : {final_preds.mean():.2f}')
print(f'Train target mean     : {y.mean():.2f}  (should be close)')

# ── 7. Save submission ────────────────────────────────────────────────────────
submission = pd.DataFrame({'Id': test['Id'], 'target': final_preds})
submission.to_csv('submission.csv', index=False)   # index=False: don't write row numbers

print()
print('Saved: submission.csv')
print(f'Rows : {len(submission)}')
print()
submission.head(10)

---
## SECTION 9 — Summary



